# Exploratory Data Analysis

Three datasets:
1. **News Category Dataset** (HuffPost, ~210k articles) — Kaggle: `rmisra/news-category-dataset`
2. **Million Headlines** (ABC Australia, ~1M headlines) — Kaggle: `therohk/million-headlines`
3. **EMDAT** — Global disaster events database (local CSV)

## Setup

Run this cell first. It installs `kagglehub` if needed and imports all libraries.

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["kagglehub", "pandas", "matplotlib", "seaborn", "openpyxl"]:
    install(pkg)

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("All libraries loaded.")

---
## Dataset 1: News Category Dataset (HuffPost)

~210,000 news articles with categories, headlines, short descriptions, and dates (2012–2022).  
Kaggle: `rmisra/news-category-dataset`

In [ ]:
# Download dataset — first run may take a minute
path1 = kagglehub.dataset_download("rmisra/news-category-dataset")
print("Downloaded to:", path1)

# The file is a JSONL (one JSON object per line)
jsonl_file = [f for f in os.listdir(path1) if f.endswith(".json")][0]
with open(os.path.join(path1, jsonl_file), "r", encoding="utf-8") as f:
    news = pd.DataFrame([json.loads(line) for line in f])

print(f"Shape: {news.shape}")
news.head(3)

In [ ]:
# Basic info
print("=== Data Types & Non-Null Counts ===")
news.info()

print("\n=== Missing Values ===")
print(news.isnull().sum())

print("\n=== Duplicate Rows ===")
print(news.duplicated().sum())

In [ ]:
# Parse date and look at time coverage
news["date"] = pd.to_datetime(news["date"], errors="coerce")
print("Date range:", news["date"].min(), "→", news["date"].max())

# Articles per year
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

news["date"].dt.year.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Articles per Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")

# Top 15 categories
news["category"].value_counts().head(15).plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Top 15 Categories")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Headline length distribution
news["headline_len"] = news["headline"].str.split().str.len()
print(news["headline_len"].describe())

news["headline_len"].plot(kind="hist", bins=40, color="steelblue", edgecolor="white")
plt.title("Distribution of Headline Word Count (Dataset 1)")
plt.xlabel("Words in headline")
plt.ylabel("Frequency")
plt.show()

---
## Dataset 2: Million Headlines (ABC Australia)

~1 million news headlines published by the Australian Broadcasting Corporation (2003–2021).  
Kaggle: `therohk/million-headlines`  
Columns: `publish_date`, `headline_text`

In [ ]:
# Download dataset
path2 = kagglehub.dataset_download("therohk/million-headlines")
print("Downloaded to:", path2)

csv_file = [f for f in os.listdir(path2) if f.endswith(".csv")][0]
headlines = pd.read_csv(os.path.join(path2, csv_file))

print(f"Shape: {headlines.shape}")
headlines.head(3)

In [ ]:
print("=== Missing Values ===")
print(headlines.isnull().sum())

print("\n=== Duplicates ===")
print(headlines.duplicated().sum())

# Parse date (format: YYYYMMDD integer)
headlines["publish_date"] = pd.to_datetime(headlines["publish_date"].astype(str), format="%Y%m%d", errors="coerce")
print(f"\nDate range: {headlines['publish_date'].min()} → {headlines['publish_date'].max()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Articles per year
headlines["publish_date"].dt.year.value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color="teal"
)
axes[0].set_title("Headlines per Year (ABC Australia)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")

# Headline word count distribution
headlines["word_count"] = headlines["headline_text"].str.split().str.len()
headlines["word_count"].plot(kind="hist", bins=30, ax=axes[1], color="teal", edgecolor="white")
axes[1].set_title("Headline Word Count Distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print(headlines["word_count"].describe())

---
## Dataset 3: EMDAT — Global Disaster Events

The Emergency Events Database tracks natural and technological disasters worldwide.  
Local file: `data/EMDAT.csv` (semicolon-delimited)  
Key columns: Disaster Type, Country, Region, Start Year, Total Deaths, Total Affected, Total Damage

In [ ]:
emdat = pd.read_csv("data/EMDAT.csv", sep=";", low_memory=False)

print(f"Shape: {emdat.shape}")
emdat.head(3)

In [ ]:
print("=== Columns ===")
print(list(emdat.columns))

print("\n=== Data Types ===")
emdat.info()

print("\n=== Missing Values (%) ===")
missing = (emdat.isnull().sum() / len(emdat) * 100).sort_values(ascending=False)
print(missing[missing > 0].round(1))

In [ ]:
# Fix numeric columns that may have commas as decimal separators
for col in ["Total Deaths", "No. Affected", "Total Affected",
            "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)"]:
    if col in emdat.columns:
        emdat[col] = (
            emdat[col].astype(str)
            .str.replace(",", ".", regex=False)
            .str.replace(" ", "", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )

print(emdat[["Start Year", "Total Deaths", "Total Affected",
             "Total Damage ('000 US$)"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Disaster type counts
emdat["Disaster Type"].value_counts().head(12).plot(
    kind="barh", ax=axes[0], color="firebrick"
)
axes[0].set_title("Top Disaster Types")
axes[0].set_xlabel("Event Count")
axes[0].invert_yaxis()

# Events per year
emdat["Start Year"].value_counts().sort_index().plot(
    kind="line", ax=axes[1], color="firebrick", marker="o", markersize=3
)
axes[1].set_title("Disaster Events per Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 countries by event count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

emdat["Country"].value_counts().head(15).plot(
    kind="barh", ax=axes[0], color="darkorange"
)
axes[0].set_title("Top 15 Countries by Event Count")
axes[0].set_xlabel("Events")
axes[0].invert_yaxis()

# Deaths by disaster type (sum, log scale)
deaths_by_type = (
    emdat.groupby("Disaster Type")["Total Deaths"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
deaths_by_type.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Total Deaths by Disaster Type (Top 10)")
axes[1].set_ylabel("Total Deaths")
axes[1].set_yscale("log")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()